In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (
        (candidate / "src" / "transform.py").is_file()
        and (candidate / "data" / "creditcard.csv").is_file()
    ):
        PROJECT_ROOT = candidate
        break
os.chdir(PROJECT_ROOT)

import pandas as pd
import tensorflow as tf

# Import Komponen Spesifik Keras untuk Arsitektur ANN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.metrics import Recall, AUC
from tensorflow.keras.optimizers import Adam

# Import Komponen Utama TFX
from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Tuner,
    Trainer,
    Evaluator,
    Pusher
)
from tfx.proto import trainer_pb2

# Import Komponen Tambahan TFX (Resolver)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

# Import Orchestration (InteractiveContext & Apache Beam)
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

# Import TensorFlow Transform & Analysis
import tensorflow_transform as tft
import tensorflow_model_analysis as tfma

print(f"Project root: {PROJECT_ROOT}")
print(f"Versi Python: {sys.version.split()[0]}")
print(f"Versi TensorFlow: {tf.__version__}")

Project root: c:\Users\ridho\tfx-fraud-detection
Versi Python: 3.9.13
Versi TensorFlow: 2.10.1


### Menyiapkan library dan komponen pipeline

Output versi Python dan TensorFlow menjadi pengecekan awal bahwa notebook berjalan pada lingkungan yang sesuai. Tidak ada proses analisis data di sini, tetapi keberhasilan impor semua komponen menunjukkan fondasi untuk menjalankan pipeline TFX dan melatih model sudah tersedia.

In [3]:
USERNAME_DICODING = "rudy_wijaya_7Xnd"
PIPELINE_NAME = f"{USERNAME_DICODING}-pipeline"
DATA_ROOT = "data"

os.makedirs(DATA_ROOT, exist_ok=True)

print(f"Directory Data: {DATA_ROOT}")
print(f"Directory Pipeline TFX: {PIPELINE_NAME}")

Directory Data: data
Directory Pipeline TFX: rudy_wijaya_7Xnd-pipeline


### Menentukan lokasi data dan pipeline

Output `Directory Data: data` dan nama pipeline memastikan folder kerja yang dipakai sudah sesuai. Folder data dibuat otomatis jika belum ada, sehingga proses berikutnya tidak gagal hanya karena direktori belum tersedia.

In [4]:
# Path keluaran dan metadata untuk Pipeline Orchestrator
OUTPUT_BASE = PIPELINE_NAME
PIPELINE_ROOT = os.path.join(OUTPUT_BASE, "pipeline_root")
METADATA_PATH = os.path.join(OUTPUT_BASE, "metadata.sqlite")
SERVING_MODEL_DIR = os.path.join(OUTPUT_BASE, "serving_model")

print(f"Pipeline Root: {PIPELINE_ROOT}")
print(f"Metadata Path: {METADATA_PATH}")
print(f"Serving Model Directory: {SERVING_MODEL_DIR}")

Pipeline Root: rudy_wijaya_7Xnd-pipeline\pipeline_root
Metadata Path: rudy_wijaya_7Xnd-pipeline\metadata.sqlite
Serving Model Directory: rudy_wijaya_7Xnd-pipeline\serving_model


### Menyiapkan konteks eksekusi TFX

Cell ini tidak menghasilkan ringkasan data yang tampil ke pengguna. Hasil pentingnya adalah terbentuknya pipeline root dan konteks metadata, sehingga setiap artefak dari tahap berikutnya dapat dilacak dan dipakai kembali oleh TFX.

In [5]:
# Ingest data menggunakan CsvExampleGen
example_gen = CsvExampleGen(input_base=DATA_ROOT)

### Mengambil data dari file CSV

Tidak adanya error saat `CsvExampleGen` dijalankan menunjukkan file CSV berhasil ditemukan dan diubah menjadi contoh data TFX. Artinya, dataset sudah berada dalam bentuk yang bisa dipakai oleh tahap statistik, validasi, dan transformasi.

In [6]:
# Inisialisasi komponen StatisticsGen
statistics_gen = StatisticsGen(
    examples=example_gen.outputs['examples']
)

### Menghasilkan statistik data

Tampilan statistik memberikan gambaran jumlah baris, tipe fitur, rentang nilai, dan distribusi label. Pada dataset kartu kredit, kelas fraud biasanya jauh lebih sedikit daripada transaksi normal; ketimpangan ini menjelaskan mengapa evaluasi tidak cukup hanya memakai accuracy dan mengapa model nantinya diberi bobot kelas.

In [7]:
# Inisialisasi komponen SchemaGen
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)

### Membuat skema dataset

Skema yang tampil menjadi kontrak struktur data untuk tahap berikutnya. Fitur seperti `V1` sampai `V28`, `Amount`, `Time`, dan label `Class` dikenali sebagai kolom numerik, sehingga pipeline memiliki aturan yang jelas saat membaca dan mengubah setiap contoh.

In [8]:
# Inisialisasi ExampleValidator
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)

### Memeriksa anomali data

Hasil validasi menunjukkan apakah data mengikuti skema yang sudah dibuat. Jika tampilan anomalies kosong atau tidak melaporkan masalah penting, struktur dataset dianggap konsisten dan proses training dapat dilanjutkan; jika ada anomali, hasil tersebut perlu diperbaiki sebelum model dipercaya.

In [ ]:
# Inisialisasi komponen Transform
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file="modules/transform.py"
)

### Mentransformasikan data

Komponen ini menghasilkan fitur dengan akhiran `_xf`, yaitu versi fitur yang sudah dinormalisasi, sekaligus menyimpan transform graph. Insight pentingnya adalah aturan preprocessing tidak hanya dipakai saat training, tetapi bisa digunakan kembali secara konsisten ketika model menerima transaksi baru.

In [ ]:
from tfx.components import Tuner
from tfx.proto import trainer_pb2

# Inisialisasi komponen Tuner
tuner = Tuner(
    module_file="modules/tuner.py",
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(num_steps=200),
    eval_args=trainer_pb2.EvalArgs(num_steps=100),
)

### Menyetel hyperparameter secara otomatis

Komponen `Tuner` mencoba beberapa kombinasi jumlah neuron, dropout, dan learning rate menggunakan PR-AUC sebagai objective. Artefak `best_hyperparameters` kemudian menjadi input eksplisit bagi `Trainer`, sehingga model yang dilatih mengikuti hasil pencarian, bukan konfigurasi statis.

In [ ]:
from tfx.components import Trainer

# Melatih model menggunakan hyperparameter terbaik dari komponen Tuner
trainer = Trainer(
    module_file="modules/trainer.py",
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(num_steps=1000),
    eval_args=trainer_pb2.EvalArgs(num_steps=500)
)

### Melatih model deteksi fraud

Training dijalankan selama 1.000 langkah dan evaluasi selama 500 langkah. Nilai loss, recall, dan AUC-PR yang muncul di log menunjukkan apakah model semakin mampu membedakan transaksi fraud dari transaksi normal; recall menjadi perhatian utama karena model yang terlalu sering melewatkan fraud tidak cocok untuk tujuan deteksi ini.

In [13]:
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

# Menginisialisasi komponen Resolver untuk mendeteksi baseline model
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
)

### Mencari model pembanding

Resolver mengambil model blessed terbaru jika ada. Bila belum ada model sebelumnya, ketiadaan baseline adalah hal yang wajar pada pelatihan pertama; model baru tetap bisa dievaluasi berdasarkan batas metrik yang sudah ditentukan.

In [15]:
import tensorflow_model_analysis as tfma
from tfx.components import Evaluator

# Konfigurasi evaluasi model dan kriteria batas minimal (threshold)
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            label_key="Class_xf"
        )
    ],
    slicing_specs=[
        tfma.SlicingSpec()
    ],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name="ExampleCount"),
                tfma.MetricConfig(class_name="AUC", config='{"curve": "PR"}'),
                tfma.MetricConfig(
                    class_name="Recall",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.90}
                        )
                    )
                ),
                tfma.MetricConfig(class_name="Precision"),
                tfma.MetricConfig(class_name="BinaryAccuracy")
            ]
        )
    ]
)

# Inisialisasi komponen Evaluator dengan memasukkan baseline model dari resolver
evaluator = Evaluator(
    examples=transform.outputs['transformed_examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)

### Mengevaluasi kelayakan model

Evaluator memeriksa AUC-PR, precision, accuracy, dan recall pada data evaluasi. Syarat recall minimal 0,90 berarti model harus menemukan setidaknya sebagian besar transaksi fraud yang sebenarnya; jika syarat ini gagal, model tidak akan diberi status blessed dan tidak boleh dipromosikan ke serving.

In [16]:
from tfx.components import Pusher
from tfx.proto import pusher_pb2

# Menentukan lokasi direktori ekspor akhir untuk model yang berstatus Blessed
SERVING_MODEL_DIR = "serving_model/fraud_detection"

# Inisialisasi komponen Pusher
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)

### Menjalankan TensorFlow Serving

Pusher menghasilkan SavedModel berversi di `serving_model/fraud_detection`. Deployment dijalankan oleh TensorFlow Serving melalui `deployment/docker-compose.serving.yml`, bukan Flask. Dari root project, jalankan `docker compose -f deployment/docker-compose.serving.yml up -d`, lalu gunakan notebook `notebooks/rudy_wijaya_7Xnd-testing.ipynb` untuk mengirim prediction request ke REST API port 8501.

In [17]:
from pathlib import Path

serving_versions = sorted(Path(SERVING_MODEL_DIR).glob("*"))
if not serving_versions:
    raise FileNotFoundError("Belum ada SavedModel hasil Pusher untuk TensorFlow Serving.")

latest_serving_version = max(serving_versions, key=lambda path: path.stat().st_mtime)
print(f"SavedModel terbaru: {latest_serving_version}")
print("Jalankan deployment dari root project:")
print("docker compose -f deployment/docker-compose.serving.yml up -d")

SavedModel terbaru: serving_model\fraud_detection\1788959970
Jalankan deployment dari root project:
docker compose -f deployment/docker-compose.serving.yml up -d


### Mengekspor model yang sudah disetujui

Folder `serving_model/fraud_detection` menjadi tanda bahwa model sudah melewati gerbang evaluasi dan siap dipakai aplikasi. Jika proses Pusher tidak menghasilkan model baru, kemungkinan besar model belum memenuhi threshold atau hasil evaluasinya belum berstatus blessed.

In [18]:
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration import pipeline
from tfx.orchestration import metadata
import os

# Nama pipeline dan direktori output sesuai ketentuan
PIPELINE_NAME = "rudy_wijaya_7Xnd-pipeline"
PIPELINE_ROOT = os.path.join(".", PIPELINE_NAME)
METADATA_PATH = os.path.join(".", PIPELINE_NAME, "metadata.sqlite")

def create_pipeline(pipeline_name: str, pipeline_root: str, metadata_path: str):
    # Kumpulkan semua komponen yang sudah diinisialisasi
    components = [
        example_gen,
        statistics_gen,
        schema_gen,
        example_validator,
        transform,
        tuner,
        trainer,
        model_resolver,
        evaluator,
        pusher
    ]

    return pipeline.Pipeline(
        pipeline_name=pipeline_name,
        pipeline_root=pipeline_root,
        metadata_connection_config=metadata.sqlite_metadata_connection_config(metadata_path),
        components=components
    )

# Jalankan pipeline menggunakan BeamDagRunner
BeamDagRunner().run(create_pipeline(PIPELINE_NAME, PIPELINE_ROOT, METADATA_PATH))

Trial 4 Complete [00h 00m 30s]
val_pr_auc: 0.7230833768844604

Best val_pr_auc So Far: 0.7230833768844604
Total elapsed time: 00h 01m 22s
Results summary
Results in .\rudy_wijaya_7Xnd-pipeline\Tuner\.system\executor_execution\27\.temp\27\fraud_detection
Showing 10 best trials
Objective(name="val_pr_auc", direction="max")

Trial 3 summary
Hyperparameters:
hidden_units: 64
dropout: 0.1
learning_rate: 0.001
Score: 0.7230833768844604

Trial 2 summary
Hyperparameters:
hidden_units: 128
dropout: 0.4
learning_rate: 0.0003
Score: 0.7230212092399597

Trial 0 summary
Hyperparameters:
hidden_units: 64
dropout: 0.30000000000000004
learning_rate: 0.001
Score: 0.7176493406295776

Trial 1 summary
Hyperparameters:
hidden_units: 32
dropout: 0.2
learning_rate: 0.0001
Score: 0.5399500131607056


Epoch 1/5
1000/1000 [==============================] - 7s 6ms/step - loss: 1.2302 - recall: 0.6835 - pr_auc: 0.4986 - val_loss: 0.1043 - val_recall: 0.9500 - val_pr_auc: 0.7501
Epoch 2/5
1000/1000 [==============================] - 5s 5ms/step - loss: 0.9066 - recall: 0.8247 - pr_auc: 0.5041 - val_loss: 0.2199 - val_recall: 0.9500 - val_pr_auc: 0.7488
Epoch 3/5
1000/1000 [==============================] - 28s 28ms/step - loss: 0.3440 - recall: 0.8590 - pr_auc: 0.6421 - val_loss: 0.1813 - val_recall: 0.9625 - val_pr_auc: 0.7243
Epoch 4/5
1000/1000 [==============================] - 33s 33ms/step - loss: 0.4480 - recall: 0.8993 - pr_auc: 0.6380 - val_loss: 0.1488 - val_recall: 0.9500 - val_pr_auc: 0.7539
Epoch 5/5
1000/1000 [==============================] - 6s 6ms/step - loss: 0.4754 - recall: 0.8878 - pr_auc: 0.4901 - val_loss: 0.1952 - val_recall: 0.9750 - val_pr_auc: 0.7121
INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:Assets written to: .\rudy_wijaya_7Xnd-pipeline\Trainer\model\28\Format-Serving\assets


INFO:tensorflow:Assets written to: .\rudy_wijaya_7Xnd-pipeline\Trainer\model\28\Format-Serving\assets


c:\Users\ridho\tfx-fraud-detection\vmenv\lib\site-packages\tensorflow_model_analysis\metrics\confusion_matrix_metrics.py:510: RuntimeWarning: invalid value encountered in divide
  prec_slope = dtp / np.maximum(dp, 0)
c:\Users\ridho\tfx-fraud-detection\vmenv\lib\site-packages\tensorflow_model_analysis\metrics\confusion_matrix_metrics.py:514: RuntimeWarning: divide by zero encountered in divide
  p[:num_thresholds - 1] / np.maximum(p[1:], 0), np.ones_like(p[1:]))


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


### Insight Eksekusi TFX Pipeline

Proses penjalanan seluruh komponen TFX Pipeline (`rudy_wijaya_7Xnd-pipeline`) menggunakan **BeamDagRunner** telah berhasil dieksekusi secara *end-to-end* tanpa kendala (*error*).

Berikut adalah ringkasan poin utama dari hasil eksekusi:

* **Hasil Hyperparameter Tuning (Tuner):**
  * Evaluasi dilakukan sebanyak **4 percobaan (trials)** dengan fokus optimasi pada matriks `val_pr_auc`.
  * **Trial 3** terpilih sebagai kombinasi terbaik dengan skor `val_pr_auc` sebesar **0,7230**. Konfigurasi hyperparameter optimal yang diperoleh:
    * `hidden_units`: 64
    * `dropout`: 0.1
    * `learning_rate`: 0.001

* **Performa Pelatihan Model (Trainer):**
  * Model dilatih selama 5 *epoch* menggunakan konfigurasi hyperparameter terbaik.
  * Hasil pelatihan menunjukkan performa deteksi yang solid dengan **`val_recall` mencapai 97,5%** (`0.9750`) dan **`val_pr_auc` stabil di angka 0,7121–0,7539**. Tingginya nilai *recall* sangat penting untuk kasus *fraud detection* agar meminimalkan luputnya transaksi mencurigakan (*false negative*).
  * Format *Serving Model* berhasil diekspor dan disimpan ke direktori `Trainer/model/28/Format-Serving/assets`.

* **Validasi & Peringatan (*Warnings*):**
  * *Warning* teknis yang muncul (seperti *deprecated functions*, *TF Transform checkpoint reference*, dan keterbatasan *property predicate*) merupakan peringatan internal standar dari TensorFlow/TFMA versi terbaru dan tidak memengaruhi validitas hasil pelatihan maupun proses ekspor *artifact*.

In [30]:
import os
import re
import tempfile
import zipfile
from pathlib import Path
import tensorflow as tf
from PIL import Image

# Konfigurasi Utama
username_dicoding = "rudy_wijaya_7Xnd"
zip_filename = "submission/submission2_mlops.zip"
max_zip_size_mb = 25
pipeline_archive_root = f"{username_dicoding}-pipeline"

# Daftar file satuan wajib & bonus (termasuk screenshot dan testing notebook)
include_files = [
    "app.py",
    "requirements.txt",
    "README.md",
    "Dockerfile",
    f"{username_dicoding}-deployment.png",
    f"{username_dicoding}-monitoring.png",
    f"{username_dicoding}-pylint.png",
    f"{username_dicoding}-grafana-dashboard.png",
    "notebooks/tfx_fraud_detection_pipeline.ipynb",
    f"notebooks/{username_dicoding}-testing.ipynb"
]

# Cari folder pipeline secara dinamis
pipeline_candidates = [
    path for path in Path(".").glob("*-pipeline") if path.is_dir()
]
if not pipeline_candidates:
    raise FileNotFoundError(
        "Folder pipeline dengan pola <username_dicoding>-pipeline tidak ditemukan di root directory."
    )

pipeline_source = next(
    (path for path in pipeline_candidates if path.name == pipeline_archive_root),
    pipeline_candidates[0],
)
print(f"Menggunakan folder pipeline: {pipeline_source}")

# Validasi keberadaan file wajib dasar
for file_path in include_files:
    # Gambar/opsional yang belum ada tidak membuat error fatal, tapi dicatat
    if not Path(file_path).is_file() and not file_path.endswith(('.png', 'Dockerfile')):
        raise FileNotFoundError(f"Berkas wajib tidak ditemukan: {file_path}")

def is_excluded_directory(path: Path) -> bool:
    excluded_names = {"__pycache__", "_wheels", "updated_analyzer_cache", ".system", "evaluator", "trainer"}
    return any(part in excluded_names for part in path.parts)

def payload_key(path: Path):
    try:
        relative_parts = path.relative_to(pipeline_source).parts
    except ValueError:
        return None
    if len(relative_parts) < 4 or path.suffix != ".gz":
        return None

    component = relative_parts[0]
    if component == "CsvExampleGen" and relative_parts[1] == "examples":
        payload_type = "csv_examples"
    elif component == "Transform" and relative_parts[1] == "transformed_examples":
        payload_type = "transformed_examples"
    else:
        return None

    version_match = re.fullmatch(r"\d+", relative_parts[2])
    split_name = relative_parts[3]
    if not version_match or not split_name.startswith("Split-"):
        return None

    return payload_type, split_name, int(version_match.group())

def select_latest_payloads():
    selected = {}
    for path in pipeline_source.rglob("*.gz"):
        if is_excluded_directory(path):
            continue
        key = payload_key(path)
        if key is None:
            continue
        payload_type, split_name, version = key
        selection_key = payload_type, split_name
        if selection_key not in selected or version > selected[selection_key][0]:
            selected[selection_key] = version, path
    return {key: value[1] for key, value in selected.items()}

def write_sample_tfrecord(source_path: Path, destination_path: Path, limit=1000):
    options = tf.io.TFRecordOptions(compression_type="GZIP")
    record_count = 0
    with tf.io.TFRecordWriter(str(destination_path), options=options) as writer:
        dataset = tf.data.TFRecordDataset([str(source_path)], compression_type="GZIP")
        for serialized_record in dataset.take(limit):
            writer.write(bytes(serialized_record.numpy()))
            record_count += 1
    return record_count

latest_payloads = select_latest_payloads()

os.makedirs("submission", exist_ok=True)
os.makedirs("submission/temp_img", exist_ok=True)

with tempfile.TemporaryDirectory(prefix="submission_payload_") as temp_dir:
    temp_dir_path = Path(temp_dir)
    sample_paths = {}
    
    if latest_payloads:
        for payload_key_value, source_path in latest_payloads.items():
            sample_path = temp_dir_path / (source_path.stem + "_submission_sample" + source_path.suffix)
            record_count = write_sample_tfrecord(source_path, sample_path)
            sample_paths[payload_key_value] = sample_path
            print(f"Prepared {payload_key_value[0]} {payload_key_value[1]} sample: {record_count} records")

    with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
        
        # 1. Masukkan file satuan (kompres otomatis gambar .png agar kecil)
        for file_path in include_files:
            if Path(file_path).is_file():
                if file_path.endswith('.png'):
                    try:
                        img = Image.open(file_path)
                        if img.mode in ('RGBA', 'P'):
                            img = img.convert('RGB')
                        temp_img = f"submission/temp_img/{Path(file_path).name}"
                        img.thumbnail((1280, 1280))
                        img.save(temp_img, "PNG", optimize=True, quality=80)
                        zipf.write(temp_img, file_path)
                        print(f"Added & compressed image: {file_path}")
                    except Exception:
                        zipf.write(file_path, file_path)
                        print(f"Added image (raw): {file_path}")
                else:
                    zipf.write(file_path, file_path)
                    print(f"Added file: {file_path}")
            else:
                print(f"Skipped missing optional file: {file_path}")

        # 2. Masukkan folder pipeline inti (tanpa file .gz besar, diganti sample terkelola)
        for source_path in pipeline_source.rglob("*"):
            if not source_path.is_file() or is_excluded_directory(source_path):
                continue
            if source_path.suffix == ".gz":
                continue  # Lewati file .gz mentah ukuran besar
            archive_path = Path(pipeline_archive_root) / source_path.relative_to(pipeline_source)
            zipf.write(source_path, archive_path.as_posix())

        # Masukkan sample TFRecord .gz yang sudah diperkecil
        for payload_key_value, sample_path in sample_paths.items():
            source_path = latest_payloads[payload_key_value]
            archive_path = Path(pipeline_archive_root) / source_path.relative_to(pipeline_source)
            zipf.write(sample_path, archive_path.as_posix())

        # 3. Masukkan folder modules/ (Wajib Saran 2)
        modules_path = Path("modules")
        if modules_path.is_dir():
            for source_path in modules_path.rglob("*"):
                if source_path.is_file() and "__pycache__" not in source_path.parts:
                    zipf.write(source_path, source_path.as_posix())
                    print(f"Added module file: {source_path}")

        # 4. Masukkan folder monitoring/ (Wajib Prometheus & Grafana)
        monitoring_path = Path("monitoring")
        if monitoring_path.is_dir():
            for source_path in monitoring_path.rglob("*"):
                if source_path.is_file() and "__pycache__" not in source_path.parts:
                    zipf.write(source_path, source_path.as_posix())
                    print(f"Added monitoring file: {source_path}")

# Validasi ukuran akhir
size_mb = Path(zip_filename).stat().st_size / (1024 * 1024)
print(f"\nUkuran file {zip_filename}: {size_mb:.2f} MB")

if size_mb > max_zip_size_mb:
    raise RuntimeError(f"Ukuran ZIP {size_mb:.2f} MB melebihi batas {max_zip_size_mb} MB.")

print("ZIP valid: Seluruh file kode, modul, monitoring, skrinsut terkompresi, dan struktur pipeline siap dikirim!")

Menggunakan folder pipeline: rudy_wijaya_7Xnd-pipeline
Prepared csv_examples Split-eval sample: 1000 records


Exception ignored in: <function WeakKeyDictionary.__init__.<locals>.remove at 0x000001C7F74D0310>
Traceback (most recent call last):
  File "C:\Users\ridho\AppData\Local\Programs\Python\Python39\lib\weakref.py", line 370, in remove
    def remove(k, selfref=ref(self)):
KeyboardInterrupt: 


Prepared csv_examples Split-train sample: 1000 records
Prepared transformed_examples Split-eval sample: 1000 records
Prepared transformed_examples Split-train sample: 1000 records
Added file: app.py
Added file: requirements.txt
Added file: README.md
Skipped missing optional file: Dockerfile
Added & compressed image: rudy_wijaya_7Xnd-deployment.png
Skipped missing optional file: rudy_wijaya_7Xnd-monitoring.png
Skipped missing optional file: rudy_wijaya_7Xnd-pylint.png
Added & compressed image: rudy_wijaya_7Xnd-grafana-dashboard.png
Added file: notebooks/tfx_fraud_detection_pipeline.ipynb
Added file: notebooks/rudy_wijaya_7Xnd-testing.ipynb
Added module file: modules\rudy_wijaya_7Xnd-pylint.png
Added module file: modules\trainer.py
Added module file: modules\transform.py
Added module file: modules\tuner.py
Added monitoring file: monitoring\Dockerfile
Added monitoring file: monitoring\prometheus.config
Added monitoring file: monitoring\prometheus.yml
Added monitoring file: monitoring\rudy